# Prophet — Walmart Store Sales Forecasting

## Goal
Forecast weekly `Weekly_Sales` for Store–Dept series. Metric = **WMAE** (holiday weeks weight **5**).

## Why Prophet (theory)
Prophet decomposes a series as:

$$y(t) = g(t) + s(t) + h(t) + \varepsilon_t$$

- **g(t)** trend (piecewise linear / logistic)
- **s(t)** seasonality (Fourier terms; we keep **yearly**, turn **weekly** off because data is already weekly)
- **h(t)** holiday effects (`add_country_holidays('US')` and/or competition `IsHoliday`)
- optional **regressors** (temperature, fuel, CPI, unemployment)

Classical / additive structure → good for **interpretation**, not for training thousands of series as one model.

## Protocol (leakage-safe)
1. Time holdout: last **8 weeks** of `train.csv` = validation
2. Fit on train only; predict val **without** using val `y`
3. Screen configs on Store 1 / Dept 1
4. Confirm top configs on **Subset50** (50 series)
5. Pick champion by **Subset50 WMAE** (not single-series)
6. **Submission**: refit champion on full `train.csv`, forecast `test.csv` → `submission_prophet.csv`

`test.csv` is never used for fitting or model selection — only for the final submission forecasts.


In [ ]:
#1 — setup
# Internet must be ON. pip conflict warnings (e.g. dacite) are harmless for this notebook.
!pip install -q prophet dagshub mlflow cmdstanpy

import os, warnings, logging, json
from pathlib import Path
from contextlib import contextmanager

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from prophet import Prophet
import mlflow

warnings.filterwarnings('ignore')

# Silence cmdstanpy spam ("Chain [1] start/done processing")
for _name in ('cmdstanpy', 'prophet', 'cmdstanpy.stanfit', 'cmdstanpy.model'):
    _log = logging.getLogger(_name)
    _log.setLevel(logging.CRITICAL)
    _log.propagate = False
    _log.disabled = True
try:
    import cmdstanpy
    cmdstanpy.set_logging_enabled(False)
except Exception:
    pass

NOTEBOOK_VERSION = 'prophet_v2_clean'
WORK_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
WORK_DIR.mkdir(parents=True, exist_ok=True)

USE_MLFLOW = False
try:
    import dagshub
    token = None
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('DAGSHUB_USER_TOKEN')
    except Exception:
        token = os.environ.get('DAGSHUB_USER_TOKEN')

    if token:
        # Compatible with current dagshub SDK (init has no `token=` arg)
        os.environ['DAGSHUB_USER_TOKEN'] = token
        dagshub.auth.add_app_token(token)
        dagshub.init(
            repo_owner='lshek22',
            repo_name='walmart-recruiting-store-sales-forecasting',
            mlflow=True,
        )
        USE_MLFLOW = True
        print('DagsHub/MLflow: token auth OK')
    else:
        print('DagsHub token missing — MLflow logging disabled (add DAGSHUB_USER_TOKEN secret to avoid OAuth popup)')
except Exception as e:
    print(f'MLflow disabled ({type(e).__name__}: {e})')

if USE_MLFLOW:
    mlflow.set_experiment('Prophet_Training')
else:
    # no-op context manager so later cells can still use `with mlflow.start_run(...)`
    @contextmanager
    def _noop_run(*args, **kwargs):
        class _Dummy:
            def __enter__(self): return self
            def __exit__(self, *a): return False
            def __getattr__(self, name):
                return lambda *a, **k: None
        yield _Dummy()
    mlflow.start_run = _noop_run
    mlflow.log_params = lambda *a, **k: None
    mlflow.log_param = lambda *a, **k: None
    mlflow.log_metric = lambda *a, **k: None
    mlflow.log_artifact = lambda *a, **k: None
    mlflow.log_text = lambda *a, **k: None

print(f'Ready | {NOTEBOOK_VERSION} | mlflow={USE_MLFLOW}')


In [ ]:
#2 — load + clean data
COMPETITION_SLUG = 'walmart-recruiting-store-sales-forecasting'


def find_competition_data_dir():
    preferred = f'/kaggle/input/competitions/{COMPETITION_SLUG}'
    if os.path.isdir(preferred):
        return preferred
    for root, _, files in os.walk('/kaggle/input'):
        if {'train.csv', 'train.csv.zip'} & set(files):
            return root
    local = Path('../input') / COMPETITION_SLUG
    if local.exists():
        return str(local)
    raise FileNotFoundError(
        'Competition data not found. Kaggle: Add Input → Competitions → '
        'Walmart Recruiting - Store Sales Forecasting'
    )


def read_competition_csv(data_dir, stem):
    for name in (f'{stem}.csv', f'{stem}.csv.zip'):
        path = os.path.join(data_dir, name)
        if os.path.exists(path):
            return pd.read_csv(path)
    raise FileNotFoundError(f'Missing {stem}.csv / .csv.zip in {data_dir}')


DATA_DIR = find_competition_data_dir()
print('DATA_DIR:', DATA_DIR)

train = read_competition_csv(DATA_DIR, 'train')
test = read_competition_csv(DATA_DIR, 'test')
stores = read_competition_csv(DATA_DIR, 'stores')
features = read_competition_csv(DATA_DIR, 'features')

for df in (train, test, features):
    df['Date'] = pd.to_datetime(df['Date'])

train = train.merge(stores, on='Store', how='left')
train = train.merge(features.drop(columns=['IsHoliday']), on=['Store', 'Date'], how='left')
test = test.merge(stores, on='Store', how='left')
test = test.merge(features.drop(columns=['IsHoliday']), on=['Store', 'Date'], how='left')

md_cols = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']
train[md_cols] = train[md_cols].fillna(0)
test[md_cols] = test[md_cols].fillna(0)
train['Weekly_Sales'] = train['Weekly_Sales'].clip(lower=0)

train = train.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)
test = test.sort_values(['Store', 'Dept', 'Date']).reset_index(drop=True)

print(f'Train {train.shape} | Test {test.shape} | series={train.groupby(["Store","Dept"]).ngroups}')
print('Note: test.csv loaded for schema only — never used for fit / selection.')


In [ ]:
#3 — split BEFORE any modeling (no leakage)
def wmae(y_true, y_pred, is_holiday):
    w = np.where(np.asarray(is_holiday).astype(bool), 5, 1)
    return float(np.sum(w * np.abs(np.asarray(y_true) - np.asarray(y_pred))) / np.sum(w))


VAL_WEEKS = 8
SPLIT_DATE = train['Date'].max() - pd.Timedelta(weeks=VAL_WEEKS)
train_set = train[train['Date'] <= SPLIT_DATE].copy()
val_set = train[train['Date'] > SPLIT_DATE].copy()

print(f'Train: {train_set.Date.min().date()} → {train_set.Date.max().date()}')
print(f'Val  : {val_set.Date.min().date()} → {val_set.Date.max().date()} ({val_set.Date.nunique()} weeks)')

with mlflow.start_run(run_name='Prophet_Setup'):
    mlflow.log_params({
        'notebook_version': NOTEBOOK_VERSION,
        'val_weeks': VAL_WEEKS,
        'split_date': str(SPLIT_DATE.date()),
        'metric': 'WMAE',
        'holiday_weight': 5,
        'leakage_protocol': 'fit_train_only_predict_val_without_y',
    })


## Helpers

`fit_predict_prophet` fits on train columns only, then predicts from val `ds` (+ regressors).  
Validation `y` is used **only** to compute WMAE — never passed into `fit` or `predict`.


In [ ]:
#4 — helpers
def to_prophet_df(df, store, dept):
    part = df[(df['Store'] == store) & (df['Dept'] == dept)].copy()
    out = part.rename(columns={'Date': 'ds', 'Weekly_Sales': 'y'})
    out['IsHoliday'] = out['IsHoliday'].fillna(0).astype(float)
    cols = ['ds', 'y', 'IsHoliday', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment']
    return out[cols].sort_values('ds').reset_index(drop=True)


def fit_predict_prophet(train_df, val_df, params, regressors=None, use_holidays=False):
    # Fit on train only; predict val without using val y (leakage-safe).
    # Re-silence logs each call (cmdstanpy sometimes re-enables handlers).
    for _name in ('cmdstanpy', 'prophet'):
        _log = logging.getLogger(_name)
        _log.setLevel(logging.CRITICAL)
        _log.disabled = True

    regressors = list(regressors or [])
    m = Prophet(**params)
    if use_holidays:
        m.add_country_holidays(country_name='US')
    for reg in regressors:
        m.add_regressor(reg)

    fit_cols = ['ds', 'y'] + regressors
    m.fit(train_df[fit_cols])

    future = val_df[['ds'] + regressors].copy()
    forecast = m.predict(future)
    preds = np.clip(forecast['yhat'].values, 0, None)
    score = wmae(val_df['y'].values, preds, val_df['IsHoliday'].values)
    return m, preds, score


BASE_PARAMS = {
    'yearly_seasonality': True,
    'weekly_seasonality': False,   # weekly data → weekly Fourier is noise
    'daily_seasonality': False,
    'seasonality_mode': 'additive',
    'changepoint_prior_scale': 0.05,
    'seasonality_prior_scale': 10.0,
    'holidays_prior_scale': 10.0,
}

STORE, DEPT = 1, 1
train_prophet = to_prophet_df(train_set, STORE, DEPT)
val_prophet = to_prophet_df(val_set, STORE, DEPT)
print(f'Screening series: Store {STORE} Dept {DEPT} | train={len(train_prophet)} val={len(val_prophet)}')


## Stage A — single-series screen (Store 1 / Dept 1)

Fast ablation to shortlist configs. Final champion is **not** chosen here.


In [ ]:
#5 — single-series screen
screen_configs = {
    'Baseline': dict(params=BASE_PARAMS.copy(), regressors=[], use_holidays=False),
    'US_Holidays': dict(params=BASE_PARAMS.copy(), regressors=[], use_holidays=True),
    'IsHoliday_Regressor': dict(params=BASE_PARAMS.copy(), regressors=['IsHoliday'], use_holidays=True),
    'Economic_Regs': dict(
        params=BASE_PARAMS.copy(),
        regressors=['Temperature', 'Fuel_Price', 'CPI', 'Unemployment'],
        use_holidays=True,
    ),
}

# slim HP variants on holiday + IsHoliday (theory: holiday weight matters most for WMAE)
_hp_flexible = {**BASE_PARAMS, 'changepoint_prior_scale': 0.5, 'seasonality_mode': 'additive'}
_hp_mult = {**BASE_PARAMS, 'changepoint_prior_scale': 0.1, 'seasonality_mode': 'multiplicative'}
screen_configs['HP_FlexibleTrend'] = dict(params=_hp_flexible, regressors=['IsHoliday'], use_holidays=True)
screen_configs['HP_Multiplicative'] = dict(params=_hp_mult, regressors=['IsHoliday'], use_holidays=True)

single_results = {}
print(f'{"Config":<22} {"WMAE":>10}')
print('-' * 34)

for name, cfg in screen_configs.items():
    _, preds, score = fit_predict_prophet(
        train_prophet, val_prophet, cfg['params'],
        regressors=cfg['regressors'], use_holidays=cfg['use_holidays'],
    )
    single_results[name] = score
    print(f'{name:<22} {score:>10,.2f}')
    with mlflow.start_run(run_name=f'Prophet_S1D1_{name}'):
        mlflow.log_params({
            **{k: str(v) for k, v in cfg['params'].items()},
            'regressors': cfg['regressors'],
            'use_holidays': cfg['use_holidays'],
            'store': STORE,
            'dept': DEPT,
            'scope': 'single_series',
        })
        mlflow.log_metric('val_wmae', score)

best_single = min(single_results, key=single_results.get)
print(f'\nBest on S1D1: {best_single} ({single_results[best_single]:,.2f})')
print('→ Next: confirm top configs on Subset50 (selection criterion).')


In [ ]:
#6 — plot best single-series forecast (demo only)
cfg = screen_configs[best_single]
m_demo, preds_demo, score_demo = fit_predict_prophet(
    train_prophet, val_prophet, cfg['params'],
    regressors=cfg['regressors'], use_holidays=cfg['use_holidays'],
)

fig, ax = plt.subplots(figsize=(12, 4))
fig.patch.set_facecolor('#0f1117')
ax.set_facecolor('#1a1d27')
ax.plot(train_prophet['ds'], train_prophet['y'], color='#8b949e', label='Train', linewidth=1)
ax.plot(val_prophet['ds'], val_prophet['y'], color='#58a6ff', label='Val actual', linewidth=2)
ax.plot(val_prophet['ds'], preds_demo, color='#f78166', label='Val forecast', linewidth=2)
ax.axvline(SPLIT_DATE, color='white', alpha=0.35, linestyle=':')
ax.set_title(f'Prophet {best_single} | Store {STORE} Dept {DEPT} | WMAE={score_demo:,.0f}', color='white')
ax.tick_params(colors='#8b949e')
ax.legend(facecolor='#1a1d27', labelcolor='white')
plt.tight_layout()
plt.savefig(str(WORK_DIR / 'prophet_s1d1_demo.png'), dpi=130, facecolor='#0f1117')
plt.show()


## Stage B — Subset50 confirmation

Evaluate the **top 3** single-series configs on 50 random Store–Dept series (≥80 train weeks).

**Important:** our 8-week holdout is **Sept–Oct 2012** (few holidays). Kaggle `test.csv` includes **Thanksgiving / Christmas**, and WMAE weights holiday weeks **5×**. So:
- lowest Subset50 score on this holdout can be a plain **Baseline**
- for **submission** we pick the best **holiday-aware** config (US holidays and/or `IsHoliday` regressor)

That is **not leakage** — it matches the competition metric / test season better.

In [ ]:
#7 — Subset50 evaluation
N_SUBSET = 50
MIN_TRAIN_WEEKS = 80
TOP_K = 3


def evaluate_prophet_subset(params, regressors, use_holidays, n_series=N_SUBSET, seed=42):
    rng = np.random.default_rng(seed)
    lengths = train_set.groupby(['Store', 'Dept']).size()
    valid_pairs = lengths[lengths >= MIN_TRAIN_WEEKS].index.tolist()
    n = min(n_series, len(valid_pairs))
    idx = rng.choice(len(valid_pairs), size=n, replace=False)
    sample = [valid_pairs[i] for i in idx]

    rows, n_failed = [], 0
    fail_reasons = {}
    for store_i, dept_i in sample:
        tr = to_prophet_df(train_set, store_i, dept_i)
        vl = to_prophet_df(val_set, store_i, dept_i)
        if len(tr) < MIN_TRAIN_WEEKS or len(vl) == 0:
            n_failed += 1
            fail_reasons['short_series'] = fail_reasons.get('short_series', 0) + 1
            continue
        try:
            _, preds, _ = fit_predict_prophet(tr, vl, params, regressors=regressors, use_holidays=use_holidays)
            for i, (_, row) in enumerate(vl.iterrows()):
                if i < len(preds):
                    rows.append({'Actual': row['y'], 'Predicted': preds[i], 'IsHoliday': row['IsHoliday']})
        except Exception as e:
            n_failed += 1
            key = type(e).__name__
            fail_reasons[key] = fail_reasons.get(key, 0) + 1

    if not rows:
        return None, n_failed, fail_reasons
    pred_df = pd.DataFrame(rows)
    score = wmae(pred_df['Actual'].values, pred_df['Predicted'].values, pred_df['IsHoliday'].values)
    return score, n_failed, fail_reasons


ranked = sorted(single_results, key=single_results.get)[:TOP_K]
# always include Baseline for a clean theory reference
if 'Baseline' not in ranked:
    ranked = ranked + ['Baseline']

subset_results = {}
print(f'Evaluating on Subset50: {ranked}')
print(f'{"Config":<22} {"Subset50 WMAE":>14} {"failed":>8}')
print('-' * 48)

for name in ranked:
    cfg = screen_configs[name]
    score, n_failed, reasons = evaluate_prophet_subset(
        cfg['params'], cfg['regressors'], cfg['use_holidays'],
    )
    subset_results[name] = score
    score_str = f'{score:,.2f}' if score is not None else 'FAIL'
    print(f'{name:<22} {score_str:>14} {n_failed:>8}  {reasons}')
    with mlflow.start_run(run_name=f'Prophet_Subset50_{name}'):
        mlflow.log_params({
            **{k: str(v) for k, v in cfg['params'].items()},
            'regressors': cfg['regressors'],
            'use_holidays': cfg['use_holidays'],
            'n_subset': N_SUBSET,
            'scope': 'subset50',
            'single_series_wmae': single_results[name],
        })
        if score is not None:
            mlflow.log_metric('val_wmae_subset50', score)
        mlflow.log_metric('n_failed', n_failed)

valid_subset = {k: v for k, v in subset_results.items() if v is not None}
assert valid_subset, 'All Subset50 runs failed — check data / Prophet install'

# Best on calm Sept–Oct holdout (may underweight holidays)
best_holdout_name = min(valid_subset, key=valid_subset.get)

# Competition WMAE weights holiday weeks 5x and Kaggle test includes Thanksgiving/Christmas.
# Prefer best holiday-aware config for submission when available.
def _is_holiday_aware(name):
    cfg = screen_configs[name]
    return bool(cfg.get('use_holidays')) or ('IsHoliday' in (cfg.get('regressors') or []))

holiday_aware = {k: v for k, v in valid_subset.items() if _is_holiday_aware(k)}
if holiday_aware:
    champion_name = min(holiday_aware, key=holiday_aware.get)
    print(
        f'Holdout-best (Sept–Oct): {best_holdout_name} = {valid_subset[best_holdout_name]:,.2f}\n'
        f'Submission champion (holiday-aware): {champion_name} = {holiday_aware[champion_name]:,.2f}'
    )
else:
    champion_name = best_holdout_name
    print(f'Champion (by Subset50): {champion_name} | WMAE={valid_subset[champion_name]:,.2f}')

champion_cfg = screen_configs[champion_name]
champion_subset_wmae = valid_subset[champion_name]


In [ ]:
#8 — comparison chart (Subset50 only) + save best
fig, ax = plt.subplots(figsize=(10, 4))
fig.patch.set_facecolor('#0f1117')
ax.set_facecolor('#1a1d27')
names = list(valid_subset.keys())
scores = [valid_subset[k] for k in names]
colors = ['#8b949e'] * len(names)
if champion_name in names:
    colors[names.index(champion_name)] = '#f78166'
bars = ax.bar(names, scores, color=colors, width=0.55)
ax.bar_label(bars, labels=[f'{v:,.1f}' for v in scores], color='white', fontsize=9)
ax.set_title(f'Prophet Subset50 WMAE — submission champion: {champion_name}', color='white')
ax.tick_params(colors='#8b949e', axis='x', rotation=15)
plt.tight_layout()
chart_path = WORK_DIR / 'prophet_subset50.png'
plt.savefig(str(chart_path), dpi=130, facecolor='#0f1117')
plt.show()

# demo refit on S1D1 with champion config (artifact only — not a global model)
champ_model, champ_preds, champ_single = fit_predict_prophet(
    train_prophet, val_prophet,
    champion_cfg['params'],
    regressors=champion_cfg['regressors'],
    use_holidays=champion_cfg['use_holidays'],
)

spec = {
    'notebook_version': NOTEBOOK_VERSION,
    'champion_config': champion_name,
    'holdout_best_config': best_holdout_name,
    'params': champion_cfg['params'],
    'regressors': champion_cfg['regressors'],
    'use_holidays': champion_cfg['use_holidays'],
    'selection_criterion': 'holiday_aware_subset50_val_wmae',
    'subset50_wmae': float(champion_subset_wmae),
    's1d1_wmae': float(champ_single),
    'single_series_screen': {k: float(v) for k, v in single_results.items()},
    'subset50_scores': {k: (float(v) if v is not None else None) for k, v in subset_results.items()},
    'note': (
        'Holdout is Sept–Oct (few holidays). Submission uses best holiday-aware '
        'config among Subset50 candidates because Kaggle test + WMAE are holiday-heavy. '
        'Classical Prophet is fit per Store-Dept series.'
    ),
}
spec_path = WORK_DIR / 'prophet_best_pipeline_spec.json'
with open(spec_path, 'w', encoding='utf-8') as f:
    json.dump(spec, f, indent=2)

with mlflow.start_run(run_name='Prophet_Best'):
    mlflow.log_params({
        'champion_config': champion_name,
        'holdout_best_config': best_holdout_name,
        'regressors': champion_cfg['regressors'],
        'use_holidays': champion_cfg['use_holidays'],
        'seasonality_mode': champion_cfg['params'].get('seasonality_mode'),
        'changepoint_prior_scale': champion_cfg['params'].get('changepoint_prior_scale'),
        'selection_criterion': 'holiday_aware_subset50_val_wmae',
    })
    mlflow.log_metric('val_wmae_subset50', champion_subset_wmae)
    mlflow.log_metric('val_wmae_s1d1', champ_single)
    mlflow.log_artifact(str(spec_path), 'pipeline')
    mlflow.log_artifact(str(chart_path), 'plots')
    mlflow.log_artifact(str(WORK_DIR / 'prophet_s1d1_demo.png'), 'plots')

print(json.dumps(spec, indent=2))
print(f'Champion ready: {champion_name} | Subset50 WMAE={champion_subset_wmae:,.2f}')
print('Next cell builds Kaggle submission.csv (refit on FULL train history).')


## Stage C — Kaggle submission

Refit the **Subset50 champion** on the **full** `train.csv` history (train + val) for every `(Store, Dept)` in `test.csv`, then forecast test dates.

- Format: `Id = Store_Dept_YYYY-MM-DD`, `Weekly_Sales`
- Failures / cold-start → last-4-week mean (else global median)
- Runtime: ~30–90 min on CPU for ~3k series (Prophet is slow per series)

Set `SUBMISSION_MAX_SERIES = 50` first if you only want a smoke test; use `None` for the real upload.

In [ ]:
#9 — build submission.csv (Prophet champion)
MAKE_SUBMISSION = True
SUBMISSION_MAX_SERIES = None  # None = all test series | 50 = quick smoke test

GLOBAL_MEDIAN = float(train['Weekly_Sales'].median())


def fallback_pred(hist_y, n):
    if len(hist_y) >= 1:
        base = float(np.mean(hist_y[-min(4, len(hist_y)):]))
    else:
        base = GLOBAL_MEDIAN
    return np.full(n, max(base, 0.0), dtype=float)


def predict_prophet_test_series(store_i, dept_i, params, regressors, use_holidays):
    hist = to_prophet_df(train, store_i, dept_i)  # FULL history (no leakage: selection already done)
    te = test[(test['Store'] == store_i) & (test['Dept'] == dept_i)].copy()
    if len(te) == 0:
        return []
    te = te.sort_values('Date')
    future = te.rename(columns={'Date': 'ds'})
    future['IsHoliday'] = future['IsHoliday'].fillna(0).astype(float)
    for col in ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment']:
        if col not in future.columns:
            future[col] = np.nan
        future[col] = future[col].ffill().bfill()

    if len(hist) < 20:
        preds = fallback_pred(hist['y'].values if len(hist) else np.array([]), len(te))
        return list(zip(te['Date'].values, preds, [True] * len(te)))  # True = fallback

    try:
        m = Prophet(**params)
        if use_holidays:
            m.add_country_holidays(country_name='US')
        for reg in regressors:
            m.add_regressor(reg)
        # keep submission logs quiet
        for _name in ('cmdstanpy', 'prophet'):
            logging.getLogger(_name).disabled = True
        m.fit(hist[['ds', 'y'] + list(regressors)])
        fc = m.predict(future[['ds'] + list(regressors)])
        preds = np.clip(fc['yhat'].values, 0, None)
        return list(zip(te['Date'].values, preds, [False] * len(te)))
    except Exception:
        preds = fallback_pred(hist['y'].values, len(te))
        return list(zip(te['Date'].values, preds, [True] * len(te)))


if MAKE_SUBMISSION:
    pairs = test.groupby(['Store', 'Dept']).size().index.tolist()
    if SUBMISSION_MAX_SERIES is not None:
        pairs = pairs[: int(SUBMISSION_MAX_SERIES)]
        print(f'SMOKE TEST: forecasting {len(pairs)} series only')
    else:
        print(f'FULL submission: forecasting {len(pairs)} series with config={champion_name}')

    rows = []
    n_fallback = 0
    for i, (store_i, dept_i) in enumerate(pairs, 1):
        out = predict_prophet_test_series(
            store_i, dept_i,
            champion_cfg['params'],
            champion_cfg['regressors'],
            champion_cfg['use_holidays'],
        )
        for dt, pred, used_fb in out:
            rows.append({
                'Id': f"{store_i}_{dept_i}_{pd.Timestamp(dt).strftime('%Y-%m-%d')}",
                'Weekly_Sales': float(pred),
            })
            n_fallback += int(used_fb)
        if i % 100 == 0 or i == len(pairs):
            print(f'  {i}/{len(pairs)} series done | fallback_rows={n_fallback}')

    # If smoke-tested a subset, still emit only those Ids (do not upload smoke file as final)
    submission = pd.DataFrame(rows)
    # Ensure full test coverage when doing full run
    if SUBMISSION_MAX_SERIES is None:
        needed = (
            test['Store'].astype(str) + '_' + test['Dept'].astype(str) + '_' + test['Date'].dt.strftime('%Y-%m-%d')
        )
        got = set(submission['Id'])
        missing = [i for i in needed if i not in got]
        if missing:
            print(f'Filling {len(missing)} missing Ids with global median={GLOBAL_MEDIAN:,.2f}')
            submission = pd.concat([
                submission,
                pd.DataFrame({'Id': missing, 'Weekly_Sales': GLOBAL_MEDIAN}),
            ], ignore_index=True)
        # preserve test row order
        order = pd.DataFrame({'Id': needed.values})
        submission = order.merge(submission, on='Id', how='left')

    sub_path = WORK_DIR / 'submission_prophet.csv'
    submission.to_csv(sub_path, index=False)
    print(f'Saved {sub_path} | rows={len(submission)} | fallback_rows={n_fallback}')
    print(submission.head())

    with mlflow.start_run(run_name='Prophet_Submission'):
        mlflow.log_params({
            'champion_config': champion_name,
            'n_series': len(pairs),
            'smoke_test': SUBMISSION_MAX_SERIES is not None,
            'fallback': 'last4_mean_or_global_median',
        })
        mlflow.log_metric('n_submission_rows', len(submission))
        mlflow.log_metric('n_fallback_rows', n_fallback)
        mlflow.log_artifact(str(sub_path), 'submission')

    print('\\nUpload submission_prophet.csv to Kaggle → Submit Predictions')
    if SUBMISSION_MAX_SERIES is not None:
        print('WARNING: this was a smoke test — set SUBMISSION_MAX_SERIES = None for a real submission.')
else:
    print('MAKE_SUBMISSION=False — skipped')
